In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from cdt.data import AcyclicGraphGenerator
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils
from causallearn.graph.GeneralGraph import GeneralGraph
from collections import Counter
from pathlib import Path
import os

In [8]:
project_root = Path('./').resolve().parents[2]
base_dir = os.path.join(project_root, 'mastersResearch_Jun_Sept2025', 'causalDiscovery_J')
print(base_dir)

/Users/joshparchure/pyspocClone5Aug/py-spoc/working_analyses/mastersResearch_Jun_Sept2025/causalDiscovery_J


In [9]:
chain_folder = os.path.join(base_dir, 'dataGen', 'chain100data')
loop_folder = os.path.join(base_dir, 'dataGen', 'loop100data')

In [10]:
# def run_pc_on_folder(folder_path):
#     """Run PC algorithm on all datasets in a folder."""
    
#     adjacency_matrices = []
#     files = [f for f in os.listdir(folder_path)]
    
#     for i, filename in enumerate(files):
#         filepath = os.path.join(folder_path, filename)
        
#         df = pd.read_csv(filepath)
#         data = df.to_numpy()
#         cg = pc(data)
#         adjacency_matrices.append(cg.G.graph)
        
#         if (i + 1) % 10 == 0:
#             print(f"  Processed {i + 1}/{len(files)} files")
    
#     return adjacency_matrices

In [11]:
def run_pc_on_folder(folder_path, max_files=100):
    """Run PC algorithm on up to `max_files` datasets in a folder."""
    adjacency_matrices = []
    graph_objects = []
    files = sorted(os.listdir(folder_path))[:max_files]  # limit to max_files

    for i, filename in enumerate(files):
        filepath = os.path.join(folder_path, filename)
        df = pd.read_csv(filepath)
        data = df.to_numpy()
        
        cg = pc(data)
        adjacency_matrices.append(cg.G.graph.copy())  # for later comparison
        graph_objects.append(cg.G)                    # for drawing
        
        if (i + 1) % 5 == 0:
            print(f"  Processed {i + 1}/{len(files)} files")
    
    return adjacency_matrices, graph_objects

In [12]:
def group_graphs_by_structure(graphs):
    """
    Groups causal-learn Graph objects by their adjacency matrix structure.
    Returns a dict: {matrix_tuple: [Graph, Graph, ...]}
    """
    grouped = {}
    for G in graphs:
        adj = tuple(map(tuple, G.graph))  # hashable matrix
        if adj not in grouped:
            grouped[adj] = []
        grouped[adj].append(G)
    return grouped


# --- Print distribution summary ---
def print_dag_distribution(dag_groups, label=''):
    print(f"\nDAG distribution for {label}:")
    for i, (adj, graphs) in enumerate(dag_groups.items(), 1):
        print(f"  DAG {i}: {len(graphs)} occurrences")


# --- Save PNG images of example DAGs ---
def save_example_dags_from_groups(dag_groups, folder_name='dag_images', prefix='dag', max_examples=3):
    os.makedirs(folder_name, exist_ok=True)

    for i, (adj, graph_list) in enumerate(dag_groups.items()):
        if i >= max_examples:
            break
        G = graph_list[0]  # Take any one of them to draw
        pyd = GraphUtils.to_pydot(G)
        filename = os.path.join(folder_name, f"{prefix}_{i+1}_count{len(graph_list)}.png")
        pyd.write_png(filename)
        print(f"Saved {filename}")


# === Main execution ===

# Run PC algorithm
print("Running PC on chain data...")
chain_adjs, chain_graphs = run_pc_on_folder(chain_folder, max_files=10)

print("Running PC on loop data...")
loop_adjs, loop_graphs = run_pc_on_folder(loop_folder, max_files=10)

# Group DAGs by unique structure
chain_groups = group_graphs_by_structure(chain_graphs)
loop_groups = group_graphs_by_structure(loop_graphs)

# Print distribution summaries
print_dag_distribution(chain_groups, label='Chain')
print_dag_distribution(loop_groups, label='Loop')

# Save example DAG visualizations
save_example_dags_from_groups(chain_groups, folder_name='chain_dags', prefix='chain')
save_example_dags_from_groups(loop_groups, folder_name='loop_dags', prefix='loop')

Running PC on chain data...


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  Processed 10/10 files

DAG distribution for Chain:
  DAG 1: 1 occurrences
  DAG 2: 1 occurrences
  DAG 3: 1 occurrences
  DAG 4: 1 occurrences
  DAG 5: 1 occurrences
  DAG 6: 1 occurrences
  DAG 7: 1 occurrences
  DAG 8: 1 occurrences
  DAG 9: 1 occurrences
  DAG 10: 1 occurrences

DAG distribution for Loop:
  DAG 1: 1 occurrences
  DAG 2: 1 occurrences
  DAG 3: 1 occurrences
  DAG 4: 1 occurrences
  DAG 5: 1 occurrences
  DAG 6: 1 occurrences
  DAG 7: 1 occurrences
  DAG 8: 1 occurrences
  DAG 9: 1 occurrences
  DAG 10: 1 occurrences


NameError: name 'GraphUtils' is not defined